Problem: Write a GAN
Problem Statement
Implement a Generative Adversarial Network (GAN) by completing the required sections. The GAN consists of a Generator that creates fake data and a Discriminator that classifies data as real or fake.

Requirements
Define the Generator Class:

Purpose: Generate fake data that mimics the real data distribution.
Layers:
Start with a fully connected layer to map the latent space (random noise) to a higher-dimensional space.
Use activation functions like ReLU to introduce non-linearity.
Add additional layers to process the data and refine its structure.
The final layer should output data in the target shape. Use an activation function like Tanh for scaling.
Forward Pass: Implement the forward method to pass the input through the defined layers.
Define the Discriminator Class:

Purpose: Classify data as real or fake.
Layers:
Use fully connected layers to process the input and extract features.
Apply activation functions like LeakyReLU to prevent dead neurons and stabilize training.
The final layer should output a single probability (real or fake) using a Sigmoid activation.
Forward Pass: Implement the forward method to process the input through the layers.
Train the GAN:

Alternate between training the Generator and Discriminator.
Use binary cross-entropy loss for both models.
Monitor the loss and generated samples during training.

In [ ]:
import torch
# Purpose: Import PyTorch for tensor operations.
# Theory: Provides tensor computations and autograd support.

import torch.nn as nn
# Purpose: Import neural network module for model classes.
# Theory: Defines Generator and Discriminator as nn.Module.

import torch.optim as optim
# Purpose: Import optimizers for training.
# Theory: Adam optimizer for stable GAN training.

import torch.nn.functional as F
# Purpose: Import functional module for activations.
# Theory: Provides ReLU, LeakyReLU, and Sigmoid.

import matplotlib.pyplot as plt
# Purpose: Import matplotlib for visualization.
# Theory: Plots real vs. generated data.

# Set random seed for reproducibility
torch.manual_seed(42)
# Purpose: Fix random seed for consistent results.
# Theory: Ensures reproducible data and weights.

class Generator(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        """
        Initializes the Generator.
        
        Args:
            latent_dim (int): Dimension of input noise vector.
            hidden_dim (int): Size of hidden layers.
            output_dim (int): Dimension of output data.
        """
        # Purpose: Define Generator architecture.
        # Theory: Maps noise to data space, mimicking real distribution.
        
        super().__init__()
        # Purpose: Initialize parent nn.Module class.
        # Theory: Enables module functionality.
        
        self.model = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Tanh()
        )
        # Purpose: Define sequential layers: FC -> ReLU -> FC -> ReLU -> FC -> Tanh.
        # Theory: Transforms noise to data, Tanh scales to [-1, 1].
    
    def forward(self, z):
        """
        Forward pass of the Generator.
        
        Args:
            z (Tensor): Noise tensor of shape (batch_size, latent_dim).
        
        Returns:
            Tensor: Generated data of shape (batch_size, output_dim).
        """
        # Purpose: Generate fake data from noise.
        # Theory: Passes noise through layers to produce samples.
        
        return self.model(z)
        # Purpose: Apply sequential layers.
        # Theory: Outputs fake data in target space.

class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        """
        Initializes the Discriminator.
        
        Args:
            input_dim (int): Dimension of input data.
            hidden_dim (int): Size of hidden layers.
        """
        # Purpose: Define Discriminator architecture.
        # Theory: Classifies data as real or fake.
        
        super().__init__()
        # Purpose: Initialize parent nn.Module class.
        # Theory: Enables module functionality.
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        # Purpose: Define sequential layers: FC -> LeakyReLU -> FC -> LeakyReLU -> FC -> Sigmoid.
        # Theory: LeakyReLU prevents dead neurons, Sigmoid outputs probability.
    
    def forward(self, x):
        """
        Forward pass of the Discriminator.
        
        Args:
            x (Tensor): Input data of shape (batch_size, input_dim).
        
        Returns:
            Tensor: Probability of real data, shape (batch_size, 1).
        """
        # Purpose: Classify input as real or fake.
        # Theory: Outputs probability score.
        
        return self.model(x)
        # Purpose: Apply sequential layers.
        # Theory: Produces probability in [0, 1].

# Training parameters
latent_dim = 100
hidden_dim = 128
output_dim = 2  # 2D points
batch_size = 64
num_epochs = 1000
lr = 0.0002
betas = (0.5, 0.999)
# Purpose: Define hyperparameters.
# Theory: Sets dimensions, training duration, and optimization parameters.

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
# Purpose: Set device for computation.
# Theory: Supports GPU acceleration if available.

# Initialize models
generator = Generator(latent_dim, hidden_dim, output_dim).to(device)
discriminator = Discriminator(output_dim, hidden_dim).to(device)
# Purpose: Create Generator and Discriminator instances.
# Theory: Moves models to device for computation.

# Optimizers
g_optimizer = optim.Adam(generator.parameters(), lr=lr, betas=betas)
d_optimizer = optim.Adam(discriminator.parameters(), lr=lr, betas=betas)
# Purpose: Initialize Adam optimizers.
# Theory: Optimizes model parameters with momentum.

# Loss function
criterion = nn.BCELoss()
# Purpose: Define binary cross-entropy loss.
# Theory: Measures classification error for Discriminator and Generator.

# Generate real data (2D Gaussian)
def get_real_data(batch_size):
    return torch.randn(batch_size, output_dim).to(device)
# Purpose: Generate synthetic real data from Gaussian.
# Theory: Simulates target distribution (mean=0, std=1).

# Training loop
g_losses = []
d_losses = []
# Purpose: Initialize lists to track losses.
# Theory: Monitors training progress.

for epoch in range(num_epochs):
    # Purpose: Iterate over training epochs.
    # Theory: Alternates Generator and Discriminator updates.
    
    # Generate real and fake data
    real_data = get_real_data(batch_size)
    z = torch.rand(batch_size, latent_dim).to(device) * 2 - 1  # U(-1, 1)
    fake_data = generator(z)
    # Purpose: Generate real and fake data batches.
    # Theory: Real data from Gaussian, fake from Generator.
    
    # Train Discriminator
    d_optimizer.zero_grad()
    # Purpose: Clear Discriminator gradients.
    # Theory: Prepares for new gradient computation.
    
    real_labels = torch.ones(batch_size, 1).to(device)
    fake_labels = torch.zeros(batch_size, 1).to(device)
    # Purpose: Define labels for real (1) and fake (0).
    # Theory: Targets for BCE loss.
    
    real_output = discriminator(real_data)
    d_loss_real = criterion(real_output, real_labels)
    # Purpose: Compute Discriminator loss on real data.
    # Theory: Measures ability to classify real data.
    
    fake_output = discriminator(fake_data.detach())
    d_loss_fake = criterion(fake_output, fake_labels)
    # Purpose: Compute Discriminator loss on fake data.
    # Theory: Measures ability to classify fake data, detach prevents Generator gradients.
    
    d_loss = d_loss_real + d_loss_fake
    d_loss.backward()
    d_optimizer.step()
    # Purpose: Update Discriminator parameters.
    # Theory: Minimizes combined real and fake loss.
    
    # Train Generator
    g_optimizer.zero_grad()
    # Purpose: Clear Generator gradients.
    # Theory: Prepares for new gradient computation.
    
    fake_output = discriminator(fake_data)
    g_loss = criterion(fake_output, real_labels)
    # Purpose: Compute Generator loss.
    # Theory: Maximizes Discriminator’s belief that fake data is real.
    
    g_loss.backward()
    g_optimizer.step()
    # Purpose: Update Generator parameters.
    # Theory: Improves fake data quality.
    
    # Store losses
    g_losses.append(g_loss.item())
    d_losses.append(d_loss.item())
    # Purpose: Track losses for monitoring.
    # Theory: Analyzes training stability.
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - D Loss: {d_loss.item():.3f}, G Loss: {g_loss.item():.3f}")
    # Purpose: Print progress every 100 epochs.
    # Theory: Monitors convergence (D loss ~0.693 is ideal).

# Test shapes
z = torch.rand(batch_size, latent_dim).to(device) * 2 - 1
fake_data = generator(z)
disc_output = discriminator(fake_data)
# Purpose: Test model output shapes.
# Theory: Verifies correct dimensions.

print("Generator output shape:", fake_data.shape)
assert fake_data.shape == (batch_size, output_dim), "Incorrect Generator output shape"
print("Discriminator output shape:", disc_output.shape)
assert disc_output.shape == (batch_size, 1), "Incorrect Discriminator output shape"
# Purpose: Verify shapes.
# Theory: Ensures models produce expected outputs.

print("Shape test passed!")
# Purpose: Confirm shape test success.
# Theory: Validates model architecture.

# Visualize results
real_data = get_real_data(1000).detach().cpu().numpy()
fake_data = generator(torch.rand(1000, latent_dim).to(device) * 2 - 1).detach().cpu().numpy()
# Purpose: Generate data for visualization.
# Theory: Compares real and generated distributions.

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.scatter(real_data[:, 0], real_data[:, 1], c='blue', alpha=0.5, label='Real')
plt.title("Real Data")
plt.legend()
plt.subplot(1, 2, 2)
plt.scatter(fake_data[:, 0], fake_data[:, 1], c='red', alpha=0.5, label='Generated')
plt.title("Generated Data")
plt.legend()
plt.savefig("gan_output.png")
plt.close()
# Purpose: Plot real vs. generated data.
# Theory: Visualizes quality of generated samples.

# Plot losses
plt.figure(figsize=(10, 5))
plt.plot(d_losses, label='Discriminator Loss')
plt.plot(g_losses, label='Generator Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN Training Losses")
plt.legend()
plt.savefig("gan_losses.png")
plt.close()
# Purpose: Plot loss curves.
# Theory: Monitors training stability and balance.

print("Visualization saved as gan_output.png and gan_losses.png")
# Purpose: Confirm visualization output.
# Theory: Provides visual evidence of training success.